<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_post_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review post processing

## Set up

In [1]:
# @title Dependencies

# Installations
!pip install -q pandas tqdm

# First-party imports
from google.colab import drive
import os
import json

# Third-party imports
import pandas as pd

In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR for the judged reviews
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews/judged")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the parsed llm reviews
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews/parsed")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet)
RESULTS_PATH = os.path.join(RAW_DIR, "results.parquet")
# Define the full result path (intermediate JSONL)
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "results.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/judged.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/parsed.


In [3]:
# @title Setup definitions

TARGET_COLUMN = "llm_judgement"

# Define error strings for failed review generation
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"

# Define error string for failed review comment segmentation
EXTRACTION_FAILURE = "ERROR: SEGMENT EXTRACTION FAILED"

# Define error strings for failed review comment judgement
JUDGE_GENERATION_FAILURE = "ERROR: FAILED TO GENERATE JUDGEMENT"

# Define error string for failed review comment judgement extraction
PARSING_FAILURE = "ERROR: FAILED TO PARSE JSON JUDGEMENT"

# Define expected judgement labels / JSON keys
EXPECTED_LABEL_KEYS = [
    'actionability_label',
    'grounding_specificity_label',
    'verifiability_label',
    'helpfulness_label'
]

In [4]:
# @title Data import

# Read processed llm reviews
df = pd.read_parquet(os.path.join(DATASET_DIR, "results.parquet"))

# Check df
display(df.shape)
display(df.head(3))


(619, 14)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segment_hash,config_group,llm_judgement,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,generation_successful
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,No end-to-end wall-clock training speedups are...,ce78cd3ae777,model: gpt-5-mini-2025-08-07 | temperature: na...,"{""actionability_label"": ""1"", ""grounding_specif...",5,4,4,True
1,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,Exact MVUE 2:4 implementation is computational...,1f58b1da7892,model: gpt-5-mini-2025-08-07 | temperature: na...,"{""actionability_label"": ""5"", ""grounding_specif...",5,4,4,True
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,The work assumes unbiasedness is strictly nece...,423e2e9fcc52,model: gpt-5-mini-2025-08-07 | temperature: na...,ERROR: FAILED TO GENERATE JUDGEMENT,5,4,4,True


## Define

In [7]:
# @title Parsing logic

def parsing_function(raw_llm_output: str) -> dict:
    """
    A wrapper function that tries to parse Pydantic-enforced JSON values from a target string.
    First, it checks if the raw output is a known error string and copies it.
    Second. if not, it attempts to parse the raw output directly as JSON.
    Third, if both of the above fail, it assigns a general parsing failure error.
    """
    result_dict = {key: None for key in EXPECTED_LABEL_KEYS}

    # List of known error strings that, if present, should be propagated directly.
    # These indicate upstream failures that are not parsing errors of the JSON structure itself.
    known_error_strings = [
        REVIEW_GENERATION_FAILURE,
        EXTRACTION_FAILURE,
        JUDGE_GENERATION_FAILURE,
    ]

    # 1. Check and copy if the raw output is an existing, propagated error string
    if raw_llm_output in known_error_strings:
        for key in EXPECTED_LABEL_KEYS:
            result_dict[key] = raw_llm_output
        return result_dict

    parsed_json_data = None
    try:
        # Attempt to parse directly as JSON
        parsed_json_data = json.loads(raw_llm_output)
    except json.JSONDecodeError:
        # If it's not a valid JSON, it will fall through to the general parsing failure
        pass

    # 2. If successfully parsed as JSON, extract labels
    if isinstance(parsed_json_data, dict):
        for key in EXPECTED_LABEL_KEYS:
            value = parsed_json_data.get(key)
            if value is not None:
                # Convert to string to match expected output type
                result_dict[key] = str(value)
            else:
                # If a label is missing from an otherwise valid JSON, mark as parsing failure for that label
                result_dict[key] = PARSING_FAILURE
    else:
        # 3. If not a known propagated error string and not a valid JSON, then it's a general parsing failure.
        for key in EXPECTED_LABEL_KEYS:
            result_dict[key] = PARSING_FAILURE

    return result_dict

## Run

In [8]:
# @title Execution

# Apply the parsing function to the target column
parsed_results = df[TARGET_COLUMN].apply(parsing_function)

# Convert the list of dictionaries into a dataset of new columns
parsed_df = pd.DataFrame(parsed_results.tolist())

# Concatenate the new columns with the original dataset
df_parsed = pd.concat([df, parsed_df], axis=1)

# Check and display
display(df_parsed.shape)
display(df_parsed.head(3))

(619, 18)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segment_hash,config_group,llm_judgement,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,generation_successful,actionability_label,grounding_specificity_label,verifiability_label,helpfulness_label
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,No end-to-end wall-clock training speedups are...,ce78cd3ae777,model: gpt-5-mini-2025-08-07 | temperature: na...,"{""actionability_label"": ""1"", ""grounding_specif...",5,4,4,True,1,3,2,2
1,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,Exact MVUE 2:4 implementation is computational...,1f58b1da7892,model: gpt-5-mini-2025-08-07 | temperature: na...,"{""actionability_label"": ""5"", ""grounding_specif...",5,4,4,True,5,1,X,1
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,The work assumes unbiasedness is strictly nece...,423e2e9fcc52,model: gpt-5-mini-2025-08-07 | temperature: na...,ERROR: FAILED TO GENERATE JUDGEMENT,5,4,4,True,ERROR: FAILED TO GENERATE JUDGEMENT,ERROR: FAILED TO GENERATE JUDGEMENT,ERROR: FAILED TO GENERATE JUDGEMENT,ERROR: FAILED TO GENERATE JUDGEMENT


## Transformation

In [9]:
# @title Results

# Save the long-format dataset to JSONL
df_parsed.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"Dataset saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format dataset to Parquet
df_parsed.to_parquet(RESULTS_PATH, index=False)
print(f"Dataset saved to Parquet at: {RESULTS_PATH}")

Dataset saved to JSONL at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/parsed/results.jsonl
Dataset saved to Parquet at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/parsed/results.parquet
